# Etapa 1 — Entendimento e Limpeza dos Dados

**Case Técnico — Análise de Desempenho de Produtos no Varejo**
Rede de materiais de construção · 11 lojas · 5 estados do Nordeste · 24 meses (jan/2024 – dez/2025)

## Objetivos
1. Inspecionar estrutura, tipos e qualidade de cada base
2. Documentar **decisões de tratamento com justificativa analítica**
3. Identificar inconsistências e levantar hipóteses de negócio
4. Gerar bases tratadas em Parquet para uso nas etapas seguintes

## Premissas
- Todos os CSVs vêm de **sistema legado brasileiro** → encoding `latin1`.
- `dimensao_precos` e `dim_produto` usam **vírgula como separador decimal** (padrão pt-BR).
- A **Loja 93 (Alhandra-PB)** opera como atacado/B2B; é sinalizada (`FLAG_LOJA93`) para
  permitir análises com e sem ela, sem distorcer as médias da rede física.
- Tratamento de nulos segue **direção conservadora** (ex.: estoque nulo → 0, que gera
  alerta precoce de ruptura).

> Cada bloco de tratamento abaixo é seguido de uma **célula de validação** (contagem de
> nulos, `head`, antes/depois) e de uma nota curta explicando o *porquê* da decisão.

<!-- glossario-comercial -->
## 📖 Glossário rápido (para leitura não técnica)

Esta seção traduz os termos técnicos do notebook para linguagem de negócio. Quem é de áreas como
comercial, marketing ou compras pode ler os resultados sem precisar do detalhe técnico.

| Termo | O que significa, em linguagem comercial |
|---|---|
| **SKU** | Código único de um produto. Cada item distinto do catálogo é um SKU (cor/tamanho/voltagem diferentes contam como SKUs diferentes). |
| **Par loja × produto** | Um produto específico dentro de uma loja específica — o menor nível de detalhe da análise de estoque. O mesmo produto em duas lojas são dois pares. |
| **Unidade de armazenagem** | A unidade padrão de contagem do estoque (p.ex. a caixa). Convertemos tudo para ela para não somar "3 caixas" com "5 unidades soltas" como se fossem iguais. |
| **Estoque projetado** | Estimativa do saldo de cada item, reconstruída pela conta *estoque de abertura + compras − vendas*. **Não é contagem física** de prateleira. |
| **Snapshot (foto)** | A situação do estoque num momento específico — aqui, o fim do período (dez/2025). |
| **Cobertura (dias de estoque)** | Por quantos dias o estoque atual daria conta da venda média. Pouca cobertura = risco de faltar produto em breve. |
| **Ruptura** | Item com estoque estimado zerado ou negativo — sem saldo para vender segundo o histórico disponível. |
| **Receita em risco** | Quanto de receita esses itens **já geraram no passado**. Serve para ordenar a fila de reposição (repor primeiro o que mais vende) — **não é perda prevista**. |
| **Curva ABC** | Regra de Pareto aplicada à receita: a **classe A** são os poucos produtos que somam ~80% do faturamento; B e C têm peso decrescente. |
| **Rede física × atacado/B2B (Loja 93)** | A Loja 93 vende em grande volume para outras empresas (atacado), com ticket ~20× o das demais. Por isso é mostrada à parte, para não distorcer a média das lojas de varejo. |
| **Linhas de venda (TRANSACOES)** | Cada linha de item vendido. Não é o número de cupons/notas (a base não traz id de cupom) — por isso é uma medida aproximada. |
| **Proxy** | Uma medida aproximada usada quando o dado ideal não existe na base (ex.: usar linhas de venda no lugar de número de cupons). |
| **Variação ano contra ano (YoY)** | Comparação de um período com o mesmo período do ano anterior (ex.: 2025 vs 2024), para isolar sazonalidade. |
| **Conservador (por construção)** | Na dúvida, o método assume o pior cenário (falta de estoque). Erra para **alertar a mais**, nunca a menos — escolha de segurança. |
| **Parquet** | Formato de arquivo compactado e rápido para grandes volumes de dados, usado internamente para acelerar o processamento. |
| **Skeleton (malha base)** | A grade inicial com todas as combinações loja × produto × mês, montada **antes** de preencher vendas e compras — garante que nenhum item suma da conta de estoque. |


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Localiza a raiz do repositório (funciona rodando de qualquer diretório)
_p = Path.cwd()
while not (_p / "src" / "utils.py").exists() and _p != _p.parent:
    _p = _p.parent
ROOT = _p
sys.path.insert(0, str(ROOT / "src"))

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUT       = ROOT / "outputs" / "etapa1"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| numpy", np.__version__)
print("Raiz do projeto:", ROOT)

pandas 3.0.3 | numpy 2.5.0
Raiz do projeto: C:\Projetos\Painel-Resultado


### Carregamento das bases brutas
As 8 bases (3 fatos + 5 dimensões) são lidas com `encoding="latin1"`. Os arquivos exportados
com `;` (dimensões e estoque) recebem `sep=";"`. `fato_vendas` e `fato_compras` usam a primeira
coluna como índice técnico (`index_col=0`).

In [2]:
vendas   = pd.read_csv(RAW / "fato_vendas_1.csv",            index_col=0, encoding="latin1")
compras  = pd.read_csv(RAW / "fato_compras_2.csv",           index_col=0, encoding="latin1")
estoque  = pd.read_csv(RAW / "fato_estoque_inicial_2.csv",   sep=";",     encoding="latin1")
produtos = pd.read_csv(RAW / "dim_produto_1.csv",            sep=";",     encoding="latin1")
lojas    = pd.read_csv(RAW / "dimensao_lojas_2.csv",         sep=";",     encoding="latin1")
precos   = pd.read_csv(RAW / "dimensao_precos_2.csv",        sep=";",     encoding="latin1")
unidades = pd.read_csv(RAW / "Descr_unidades_medida_2.csv",  sep=";",     encoding="latin1")
voltagem = pd.read_csv(RAW / "dimensao_voltagem_2.csv",      sep=";",     encoding="latin1")

print("Linhas por base:")
for nome, df_ in [("vendas",vendas),("compras",compras),("estoque",estoque),
                  ("produtos",produtos),("lojas",lojas),("precos",precos)]:
    print(f"  {nome:<10}: {len(df_):>9,} linhas × {df_.shape[1]} colunas")
vendas.head(3)

Linhas por base:
  vendas    : 1,090,390 linhas × 9 colunas
  compras   :     1,393 linhas × 7 colunas
  estoque   :    25,330 linhas × 3 colunas
  produtos  :     2,731 linhas × 14 colunas
  lojas     :        11 linhas × 3 colunas
  precos    :    28,560 linhas × 7 colunas


,DATA_VENDA,COD_EMPRESA,CODIGO,DIGITO,EMBALAGEM,QUANTIDADE_VENDIDA,CONVERSAO_VENDA_PARA_ARMAZENAGEM,UNIDADE_DA_VENDA,PRECO_UNIT_MEDIO
0,2024-01-02,93,101702,0,0,1.0,1.0,UN,2036.895
1,2024-01-02,6,102242,3,0,1.0,1.0,UN,35.595
2,2024-01-02,7,102242,3,0,1.0,1.0,UN,39.795


**Por quê esta etapa?** Antes de qualquer cálculo precisamos garantir que os acentos foram
lidos corretamente (latin1) e que os números decimais não viraram texto. Um único erro de
encoding ou de separador decimal se propaga por toda a análise — daí a inspeção logo no início.

In [3]:
# 2.1 — Datas para datetime
vendas["DATA_VENDA"]    = pd.to_datetime(vendas["DATA_VENDA"])
compras["DATA_ENTRADA"] = pd.to_datetime(compras["DATA_ENTRADA"])

# validação
print("vendas  DATA_VENDA  :", vendas["DATA_VENDA"].min().date(), "→", vendas["DATA_VENDA"].max().date(),
      "| dtype:", vendas["DATA_VENDA"].dtype)
print("compras DATA_ENTRADA:", compras["DATA_ENTRADA"].min().date(), "→", compras["DATA_ENTRADA"].max().date(),
      "| dtype:", compras["DATA_ENTRADA"].dtype)

vendas  DATA_VENDA  : 2024-01-02 → 2025-12-31 | dtype: datetime64[us]
compras DATA_ENTRADA: 2024-01-03 → 2025-12-17 | dtype: datetime64[us]


**Por quê?** Datas em texto impedem agrupamentos mensais (`to_period("M")`) e ordenação
temporal — essenciais para a série de estoque da Etapa 2. A conversão confirma também que o
período cobre exatamente os 24 meses esperados.

In [4]:
# 2.2 — Estoque inicial: nulos → 0  (DECISÃO conservadora)
n_antes_nulos = pd.to_numeric(estoque["ESTOQUE_INICIAL"], errors="coerce").isna().sum()

estoque["ESTOQUE_INICIAL"] = pd.to_numeric(estoque["ESTOQUE_INICIAL"], errors="coerce").fillna(0)

n_depois_nulos = estoque["ESTOQUE_INICIAL"].isna().sum()
print(f"Nulos em ESTOQUE_INICIAL — antes: {n_antes_nulos:,} ({n_antes_nulos/len(estoque)*100:.1f}%)  |  depois: {n_depois_nulos}")
print(f"Registros tratados como 0: {n_antes_nulos:,} de {len(estoque):,}")

Nulos em ESTOQUE_INICIAL — antes: 1,329 (5.2%)  |  depois: 0
Registros tratados como 0: 1,329 de 25,330


**Por quê tratar nulo como 0?** A ausência de registro de estoque indica produto **não
disponível** naquela loja no início do período. Assumir 0 é *conservador*: subestima o estoque,
o que gera **alerta precoce de ruptura** — direção segura para decisões de reposição. O oposto
(imputar uma média) inflaria o estoque e mascararia rupturas reais.

In [5]:
# 2.3 — Preços: vírgula → ponto decimal (padrão pt-BR)
amostra_antes = precos["PRECO_EMBALAGEM_0"].head(3).tolist()

for col in ["PRECO_EMBALAGEM_0","PRECO_EMBALAGEM_1","PRECO_EMBALAGEM_2",
            "PERC_DESCTO_ADICIONAL_EMBALAGEM_0"]:
    precos[col] = precos[col].astype(str).str.replace(",", ".", regex=False)
    precos[col] = pd.to_numeric(precos[col], errors="coerce")

print("PRECO_EMBALAGEM_0 — antes (texto):", amostra_antes)
print("PRECO_EMBALAGEM_0 — depois (float):", precos["PRECO_EMBALAGEM_0"].head(3).tolist())
print("dtype depois:", precos["PRECO_EMBALAGEM_0"].dtype)

PRECO_EMBALAGEM_0 — antes (texto): ['25,095', '31,395', '20,895']
PRECO_EMBALAGEM_0 — depois (float): [25.095, 31.395, 20.895]
dtype depois: float64


**Por quê?** Lidos como texto (`"12,90"`), esses campos retornariam `NaN` em qualquer operação
numérica, **inviabilizando cálculos de margem e receita**. A troca `,`→`.` seguida de
`to_numeric` recupera os valores como `float64`.

In [6]:
# 2.4 — Fator de conversão de compra (mesmo problema de vírgula)
produtos["CONVERSAO_COMPRA_ARMAZENAGEM"] = pd.to_numeric(
    produtos["CONVERSAO_COMPRA_ARMAZENAGEM"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)
print("CONVERSAO_COMPRA_ARMAZENAGEM — dtype:", produtos["CONVERSAO_COMPRA_ARMAZENAGEM"].dtype)
print(produtos["CONVERSAO_COMPRA_ARMAZENAGEM"].describe()[["min","50%","max"]])

CONVERSAO_COMPRA_ARMAZENAGEM — dtype: float64
min      0.2
50%      1.0
max    100.0
Name: CONVERSAO_COMPRA_ARMAZENAGEM, dtype: float64


**Por quê?** Esse fator converte a unidade de compra (ex.: caixa) para a unidade de
armazenagem. Um fator lido errado (texto → NaN → 1) **distorceria todo o volume de estoque**
calculado na Etapa 2 — uma caixa de 12 entraria como 1 unidade.

In [7]:
# 2.5 — Colunas derivadas em vendas
vendas["RECEITA"]         = vendas["QUANTIDADE_VENDIDA"].astype(float) * vendas["PRECO_UNIT_MEDIO"].astype(float)
vendas["QTD_ARMAZENAGEM"] = vendas["QUANTIDADE_VENDIDA"].astype(float) * vendas["CONVERSAO_VENDA_PARA_ARMAZENAGEM"].astype(float)
vendas["ANO_MES"]         = vendas["DATA_VENDA"].dt.to_period("M").astype(str)
vendas["ANO"]             = vendas["DATA_VENDA"].dt.year
vendas["MES"]             = vendas["DATA_VENDA"].dt.month
vendas["FLAG_LOJA93"]     = (vendas["COD_EMPRESA"] == 93).astype(int)

# validação
print("Receita total (24M): R$ {:.1f}M".format(vendas["RECEITA"].sum()/1e6))
print("Linhas FLAG_LOJA93=1 :", f"{int(vendas['FLAG_LOJA93'].sum()):,}")
vendas[["DATA_VENDA","COD_EMPRESA","CODIGO","QUANTIDADE_VENDIDA",
        "CONVERSAO_VENDA_PARA_ARMAZENAGEM","PRECO_UNIT_MEDIO",
        "RECEITA","QTD_ARMAZENAGEM","FLAG_LOJA93"]].head(4)

Receita total (24M): R$ 482.5M
Linhas FLAG_LOJA93=1 : 24,883


,DATA_VENDA,COD_EMPRESA,CODIGO,QUANTIDADE_VENDIDA,CONVERSAO_VENDA_PARA_ARMAZENAGEM,PRECO_UNIT_MEDIO,RECEITA,QTD_ARMAZENAGEM,FLAG_LOJA93
0,2024-01-02,93,101702,1.0,1.0,2036.895,2036.895,1.0,1
1,2024-01-02,6,102242,1.0,1.0,35.595,35.595,1.0,0
2,2024-01-02,7,102242,1.0,1.0,39.795,39.795,1.0,0
3,2024-01-02,3,102242,1.0,1.0,38.745,38.745,1.0,0


**Por quê estas 3 colunas derivadas?**
- **RECEITA** = quantidade × preço — base de todos os rankings de desempenho.
- **QTD_ARMAZENAGEM** = quantidade × conversão — sem ela, packs e caixas seriam somados como
  unidades e a posição de estoque (Etapa 2) ficaria errada.
- **FLAG_LOJA93** — permite separar o atacado da rede física em qualquer corte, sem remover dados.

In [8]:
# 2.6 — Colunas derivadas em compras
compras["VALOR_TOTAL_COMPRA"] = compras["QUANTIDADE_COMPRA"] * compras["PRECO_UNIT_UNIDADE_COMPRA"]
compras["FLAG_SEM_PRECO"]     = compras["PRECO_UNIT_UNIDADE_COMPRA"].isna().astype(int)
compras["ANO_MES"]            = compras["DATA_ENTRADA"].dt.to_period("M").astype(str)

n_sem_preco = int(compras["FLAG_SEM_PRECO"].sum())
print(f"Entradas sem preço de compra: {n_sem_preco:,} ({n_sem_preco/len(compras)*100:.1f}%)")
print("→ volume físico mantido; excluídas só do CMV e do valor total de compras.")

Entradas sem preço de compra: 132 (9.5%)
→ volume físico mantido; excluídas só do CMV e do valor total de compras.


**Por quê manter as compras sem preço?** O **volume físico** é válido para o cálculo de
estoque. Imputar um custo inexistente distorceria a margem; por isso essas 132 entradas são
sinalizadas (`FLAG_SEM_PRECO`) e excluídas **apenas** dos cálculos financeiros.

In [9]:
# 2.7 — Join principal: vendas enriquecida com produto + loja
vendas_enr = (
    vendas
    .merge(produtos[["CODIGO","DESCRICAO","NIVEL_1","NIVEL_2","NIVEL_3",
                     "UNIDADE_ESTOQUE","CONVERSAO_COMPRA_ARMAZENAGEM","CD_VOLTAGEM"]],
           on="CODIGO", how="left")
    .merge(lojas, on="COD_EMPRESA", how="left")
)
print("vendas_enr:", f"{len(vendas_enr):,} linhas × {vendas_enr.shape[1]} colunas")
vendas_enr[["CODIGO","DESCRICAO","NIVEL_1","COD_EMPRESA","CD_CIDADE","CD_ESTADO","RECEITA"]].head(4)

vendas_enr: 1,090,390 linhas × 24 colunas


,CODIGO,DESCRICAO,NIVEL_1,COD_EMPRESA,CD_CIDADE,CD_ESTADO,RECEITA
0,101702,REFRIG. 1P 261L DSECO BR CRA30 M220,D - ELETROS,93,ALHANDRA,PB,2036.895
1,102242,RESIST.DUCH.GOR.4T 5700W M220,U - METAIS E ACESSORIOS,6,JOAO PESSOA,PB,35.595
2,102242,RESIST.DUCH.GOR.4T 5700W M220,U - METAIS E ACESSORIOS,7,NATAL,RN,39.795
3,102242,RESIST.DUCH.GOR.4T 5700W M220,U - METAIS E ACESSORIOS,3,SALVADOR,BA,38.745


**Por quê o join agora?** Centralizar produto + loja em `fato_vendas` evita repetir merges em
todas as etapas seguintes e permite verificar **integridade referencial** de uma só vez (abaixo).

In [10]:
# 3 — Integridade referencial
codigos_vendas  = set(vendas["CODIGO"].unique())
codigos_dim     = set(produtos["CODIGO"].unique())
codigos_compras = set(compras["CODIGO"].unique())
lojas_vendas    = set(vendas["COD_EMPRESA"].unique())
lojas_dim       = set(lojas["COD_EMPRESA"].unique())

print(f"Produtos em vendas sem dim_produto : {len(codigos_vendas - codigos_dim)}")
print(f"Produtos em compras sem dim_produto: {len(codigos_compras - codigos_dim)}")
print(f"Lojas em vendas sem dim_lojas      : {len(lojas_vendas - lojas_dim)}")
print(f"Produtos sem nenhuma venda         : {len(codigos_dim - codigos_vendas)}")
print(f"Nulos em NIVEL_1 após join         : {int(vendas_enr['NIVEL_1'].isna().sum())}")
print(f"Nulos em CD_CIDADE após join       : {int(vendas_enr['CD_CIDADE'].isna().sum())}")

Produtos em vendas sem dim_produto : 0
Produtos em compras sem dim_produto: 0
Lojas em vendas sem dim_lojas      : 0
Produtos sem nenhuma venda         : 2
Nulos em NIVEL_1 após join         : 0
Nulos em CD_CIDADE após join       : 0


**Leitura:** integridade referencial **100%** entre vendas, produtos e lojas (zero órfãos).
Apenas **2 produtos** existem no cadastro sem nenhuma venda nos 24 meses — candidatos a
descontinuação, sinalizados mas mantidos na dimensão.

In [11]:
# 4 — Estatísticas-resumo
kpis = {
    "Receita total (24M)": f"R$ {vendas_enr['RECEITA'].sum()/1e6:,.1f}M",
    "Transações":          f"{len(vendas_enr):,}",
    "SKUs ativos":         f"{vendas_enr['CODIGO'].nunique():,}",
    "Lojas":               f"{vendas_enr['COD_EMPRESA'].nunique()}",
    "Categorias (Nível 1)":f"{vendas_enr['NIVEL_1'].nunique()}",
    "Período":             f"{vendas_enr['DATA_VENDA'].min().date()} → {vendas_enr['DATA_VENDA'].max().date()}",
}
for k,v in kpis.items():
    print(f"  {k:<22}: {v}")

# Receita por loja (com destaque da loja 93)
rl = (vendas_enr.groupby("COD_EMPRESA")["RECEITA"].sum().sort_values(ascending=False)/1e6).round(1)
print("\nReceita por loja (R$ M):")
print(rl.to_string())

  Receita total (24M)   : R$ 482.5M
  Transações            : 1,090,390
  SKUs ativos           : 2,729
  Lojas                 : 11
  Categorias (Nível 1)  : 23
  Período               : 2024-01-02 → 2025-12-31

Receita por loja (R$ M):
COD_EMPRESA
93    153.3
3      62.6
2      53.4
4      43.3
6      34.8
92     28.6
5      23.5
7      22.3
8      21.3
9      21.1
1      18.4


**Leitura dos KPIs:** ~R$ 482,5M em receita, 1,09M transações, 2.729 SKUs. A **Loja 93**
concentra ~31,8% da receita com apenas ~12% dos SKUs — confirmando o perfil atacadista que
justifica analisá-la em separado.

In [12]:
# 5 — Salvar bases tratadas em Parquet
vendas_enr.to_parquet(PROCESSED / "vendas_tratadas.parquet",        index=False)
compras.to_parquet(   PROCESSED / "compras_tratadas.parquet",        index=False)
estoque.to_parquet(   PROCESSED / "estoque_inicial_tratado.parquet", index=False)
produtos.to_parquet(  PROCESSED / "dim_produto_tratada.parquet",     index=False)
precos.to_parquet(    PROCESSED / "dim_precos_tratada.parquet",      index=False)
lojas.to_parquet(     PROCESSED / "dim_lojas.parquet",               index=False)
unidades.to_parquet(  PROCESSED / "dim_unidades.parquet",            index=False)
voltagem.to_parquet(  PROCESSED / "dim_voltagem.parquet",            index=False)
print("Bases salvas em data/processed/ (8 arquivos Parquet)")

Bases salvas em data/processed/ (8 arquivos Parquet)


## Resumo dos KPIs validados — Etapa 1

| KPI | Valor |
|---|---|
| Receita total (24 meses) | **R$ 482,5M** |
| Transações | **1.090.390** |
| SKUs ativos | **2.729** |
| Lojas | **11** (5 estados: PE, BA, SE, PB, RN) |
| Categorias (Nível 1) | **23** |
| Período | jan/2024 → dez/2025 |
| Integridade referencial | **100%** (0 órfãos em produto/loja) |
| Estoque inicial nulo → 0 | 1.329 registros (5,2%) |
| Compras sem preço | 132 entradas (9,5%) — fora do CMV |
| Loja 93 (atacado) | ~31,8% da receita · ~12% dos SKUs |

**10 decisões de tratamento** documentadas em `outputs/etapa1/decisoes_tratamento.csv`;
**35 campos** descritos em `outputs/etapa1/dicionario_dados.csv`.
As bases tratadas alimentam a **Etapa 2 (estoque projetado)**.